# Notebook 17 — ISLES2024 Dataset505 raw perfusion intensity audit

**Read-only diagnostic, not a model training or preprocessing run.** Use **Colab CPU**, then **Runtime → Run all**. It scans all 149 original five-channel nnU-Net staging NIfTI images, looking for patient-specific extreme MTT/Tmax/CBV/CBF values.

**Do not clip or normalize data here.** Negative perfusion-map values are *flags for review*, not automatically proven corruption. The nnU-Net foreground fingerprint reports MTT = −29,691,790 and unusually large standard deviation, but it does not tell us which patient has the extreme or how often it occurs. This notebook identifies those cases and quantifies the affected voxels.

Outputs stay in private Google Drive. It records 149 resumable patient checkpoints, a progress status file, a full private per-patient CSV, a small channel-level aggregate JSON, and the execution log. No patient images are uploaded to GitHub or modified.

**Scientific caveat:** This audits raw full-volume image intensities. It does *not* claim to duplicate nnU-Net's crop/mask normalization support. Any replacement normalization and Dataset506/507 experiment requires its own declared protocol and frozen original Dataset504/505 untouched.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Set paths, verify original data and download pinned analysis code

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
IMAGES = ROOT / 'nnUNet_raw' / 'Dataset505_ISLES2024_NCCT_PERFUSION' / 'imagesTr'
SPLITS = ROOT / 'nnUNet_preprocessed' / 'Dataset505_ISLES2024_NCCT_PERFUSION' / 'splits_final.json'
OUTPUT = ROOT / 'results' / 'perfusion_intensity_audit'
LOG = OUTPUT / 'perfusion_intensity_audit.log'
SCRIPT = Path('/content/audit_isles2024_perfusion_intensity.py')
SCRIPT_COMMIT = 'd2eecc86736f2befa0e318f163efa771845527ec'

OUTPUT.mkdir(parents=True, exist_ok=True)
if not IMAGES.is_dir(): raise FileNotFoundError(f'Missing raw Dataset505 imagesTr: {IMAGES}')
if not SPLITS.is_file(): raise FileNotFoundError(f'Missing immutable splits: {SPLITS}')

def run_logged(label, cmd):
    cmd = [str(s) for s in cmd]
    print('\n===', label, '===', flush=True)
    with LOG.open('a',encoding='utf8') as log:
        log.write(f'\n[{datetime.now(timezone.utc).isoformat()}] {label}: $ {" ".join(cmd)}\n')
        log.flush()
        with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED':'1'}) as p:
            for line in p.stdout:
                print(line, end='', flush=True)
                log.write(line);log.flush()
            code=p.wait()
        log.write(f'[exit_code={code}]\n');log.flush()
        if code: raise RuntimeError(f'{label} failed (exit {code}): {LOG}')

run_logged('Fetch immutable GitHub audit script', [
    'curl','-fLsS','--retry','5','--retry-delay','3',
    f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{SCRIPT_COMMIT}/scripts/audit_isles2024_perfusion_intensity.py',
    '-o',SCRIPT])
assert SCRIPT.is_file() and SCRIPT.stat().st_size > 5000
print('Audit source commit:', SCRIPT_COMMIT)
print('Private output:', OUTPUT)


## 2. Install lightweight dependencies and run the complete 149-case audit

The pinned code uses NumPy, nibabel and Python standard libraries. The per-patient checkpoints remain intact on disconnection; rerun **Run all** to resume. Min/max and extreme counts are exact for each file; reported quantiles are deterministic subsamples (maximum 1 million voxels per channel) to limit RAM.


In [ ]:
run_logged('Install CPU-only NIfTI dependencies', [sys.executable,'-m','pip','install','--disable-pip-version-check',
    '--only-binary=:all:', 'numpy>=1.26,<3', 'nibabel>=5.2,<6'])
run_logged('Audit all raw Dataset505 channels', [
    sys.executable,'-u',SCRIPT,'--images-tr',IMAGES,'--split-file',SPLITS,
    '--output-dir',OUTPUT,'--retries','3'])


## 3. Confirm full output and flag possible artifacts

The output is **descriptive**. Do not use this notebook to select clinical clipping bounds from held-out outcomes. If you propose new preprocessing, determine it from appropriate training data and report a separate controlled ablation.


In [ ]:
report = json.loads((OUTPUT/'isles2024_perfusion_intensity_audit_aggregate.json').read_text())
progress = json.loads((OUTPUT/'intensity_audit_progress.json').read_text())
assert progress['status']=='complete' and progress['completed_cases']==149, progress
assert report['n_patients']==149 and report['n_channel_volumes']==745, report
print('Audit complete:', report['n_patients'], 'patients and', report['n_channel_volumes'], 'channels')
for name, stats in report['channels'].items():
    print(f'{name}: min={stats["minimum_across_patients"]:,.2f}, '+
          f'cases below -10,000: {stats["cases_with_values_below_minus_10000"]}/149, '+
          f'nonfinite cases: {stats["cases_with_nonfinite_voxels"]}')
print('Private detailed CSV:', OUTPUT/'isles2024_perfusion_case_intensity_audit_PRIVATE.csv')
print('Aggregate JSON:', OUTPUT/'isles2024_perfusion_intensity_audit_aggregate.json')
